# CoChem-BASE: Master Environment Orchestrator

Welcome to the **CoChem-BASE** initialization matrix. This environment replaces all legacy Docker and DevContainer layers, allowing you to natively provision your exact Interaction and Calculation hardware profiles.

### 🚀 Quick Start Instructions
1. Click the code cell below and press `Shift + Enter` to Setup the Silo and Environment Paths.
2. Select your newly created `cochem_base_silo` Kernel when prompted.
3. Once selected, run the final cell to render the Matrix Dashboard.


## 💾 Silo Setup & Artifact Registry Configuration

**Purpose:**
To establish a dedicated, reproducible computational environment (Silo) and configure a persistent local directory for storing generated chemistry artifacts.

**Instructions:**
- Run the code cell below by clicking it and pressing `Shift + Enter`.
- Select **New Install** to configure a fresh Silo, or **Keep previous setup** to validate an existing one.
- Enter an artifact directory, or leave the default `CoChem_Artifacts` directory in your user home.
- Optionally map a Conda, Mamba, or Micromamba executable. A command already available on `PATH` also works.
- Click **Create & Provision** to build the Silo.
- If the build succeeds, select the new `cochem_base_silo` kernel using the kernel selector.

**Didactic Breakdown:**
In computational chemistry and chemoinformatics, exact software environments are critical. Slight dependency differences can produce irreproducible energies, broken trajectory visualization, or incompatible quantum-mechanical properties.

This setup resolves repository, artifact, and environment-manager paths dynamically. The isolated `cochem_base_silo` therefore remains reproducible when the checkout is moved between Windows, macOS, Linux, Codespaces, and mapped storage locations.


In [ ]:
import importlib
import os
import shutil
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

import cochem_base.config_loader as config_loader


def resolve_base_root():
    mapped_root = os.environ.get('COCHEM_BASE_ROOT')
    if mapped_root:
        root = Path(os.path.expandvars(mapped_root)).expanduser().resolve()
        if (root / 'setup' / 'cochem_base_setup.py').is_file():
            return root
        raise RuntimeError(f'COCHEM_BASE_ROOT does not contain CoChem-BASE: {root}')
    for candidate in (Path.cwd(), *Path.cwd().parents):
        for root in (candidate, candidate / 'CoChem-BASE'):
            if (root / 'setup' / 'cochem_base_setup.py').is_file():
                return root.resolve()
    raise RuntimeError('Unable to locate CoChem-BASE. Set COCHEM_BASE_ROOT to this checkout.')

def resolve_artifact_path(value):
    path = Path(os.path.expandvars(value)).expanduser()
    if not path.is_absolute():
        path = Path.home() / path
    return path.resolve()

BASE_ROOT = resolve_base_root()
if str(BASE_ROOT) not in sys.path:
    sys.path.insert(0, str(BASE_ROOT))
os.environ['COCHEM_BASE_ROOT'] = str(BASE_ROOT)
for module_name in tuple(sys.modules):
    if module_name in {'cochem_base', 'core_engine', 'setup'} or module_name.startswith(('cochem_base.', 'core_engine.', 'setup.')):
        sys.modules.pop(module_name, None)

config_loader = importlib.reload(config_loader)
resolve_conda_executable = config_loader.resolve_conda_executable

# 1) Detect if this setup stage has been completed.
keep_btn = widgets.Button(description="Keep previous setup", button_style="info")
new_btn = widgets.Button(description="New Install", button_style="warning")
out = widgets.Output()

def on_keep(b):
    with out:
        clear_output()
        print("Testing previous setup...")
        try:
            from test_suite.test_environment import check_artifacts_dir, check_cochem_base_silo
            silo_ok, silo_msg = check_cochem_base_silo()
            art_ok, art_msg = check_artifacts_dir()
            print(silo_msg)
            print(art_msg)
            if silo_ok and art_ok:
                print("✅ Everything is ready for the next step!")
            else:
                print("❌ Environment validation failed. Please run a New Install.")
        except Exception as e:
            print(f"❌ Error: {e}. Please run a New Install.")
            raise

def on_new(b):
    with out:
        clear_output()
        default_art = resolve_artifact_path(os.environ.get('COCHEM_ARTIFACT_DIR', 'CoChem_Artifacts'))
        path_input = widgets.Text(
            value=str(default_art),
            description='Artifacts Path:',
            style={'description_width': 'initial'}
        )
        conda_path_input = widgets.Text(
            value=os.environ.get('COCHEM_CONDA_EXE', resolve_conda_executable(required=False)),
            description='Conda/Mamba:',
            style={'description_width': 'initial'}
        )
        submit_btn = widgets.Button(description="Create & Provision", button_style="success")

        def on_submit(b2):
            with out:
                clear_output()
                target_path = resolve_artifact_path(path_input.value)
                silo_path = target_path / 'Silos'
                if silo_path.exists():
                    print(f"Deleting previous Silos directory at {silo_path}...")
                    shutil.rmtree(silo_path, ignore_errors=True)
                silo_path.mkdir(parents=True, exist_ok=True)
                print(f"Created CoChem_Artifacts/Silos at {silo_path}")
                print("Setting up minimum environment...")
                os.environ['COCHEM_ARTIFACT_DIR'] = str(target_path)
                if conda_path_input.value.strip():
                    os.environ['COCHEM_CONDA_EXE'] = conda_path_input.value.strip()
                try:
                    from setup.cochem_base_setup import setup_cochem_base
                    setup_cochem_base()
                    print("✅ New installation completed and ready for the next step!")
                except Exception as e:
                    print(f"Error during setup: {e}")
                    raise

        submit_btn.on_click(on_submit)
        display(path_input, conda_path_input, submit_btn)

keep_btn.on_click(on_keep)
new_btn.on_click(on_new)

display(widgets.HBox([keep_btn, new_btn]), out)


## 🎛️ CoChem-BASE Interactive Matrix Dashboard

**Purpose:**
To launch the primary graphical user interface used to route computational chemistry tasks, configure execution environments, and integrate external chemistry software (like ORCA).

**Instructions:**
- **If** you just provisioned the Silo in the previous cell, **then** you MUST click the kernel link provided in the success message (or use the kernel selector in the top right of VS Code) to switch your active Python kernel to `cochem_base_silo`.
- Wait exactly 2 seconds for the kernel to attach.
- Execute the code cell below by clicking it and pressing `Shift + Enter`.
- **If** an error stating "UNITY dashboard missing" appears, **then** verify you are running this notebook from the root directory of the CoChem-BASE repository.
- **If** the dashboard successfully loads, **then** you may proceed to configure your interaction and calculation hardware profiles directly via the UI.

**Didactic Breakdown:**
Executing advanced chemical computations often requires orchestrating incredibly complex workflows across diverse hardware architectures (e.g., transitioning jobs between local graphical workstations, Windows Subsystem for Linux (WSL), and High-Performance Computing (HPC) clusters).

This cell bridges the gap between high-level Jupyter notebook interaction and low-level subprocess execution. It dynamically maps and injects a custom Python-based GUI into memory, preventing pollution of the system path while surfacing a robust dashboard. This architecture empowers researchers to intuitively configure molecular dynamics simulations, electronic structure jobs, and thermodynamic modeling parameters without being forced to manually edit complex, error-prone shell scripts or JSON configuration files.

In [ ]:
import importlib
import os
import platform

import ipywidgets as widgets
from IPython.display import display

import cochem_base.config_loader as config_loader

config_loader = importlib.reload(config_loader)
get_modules_dir = config_loader.get_modules_dir
resolve_executable = config_loader.resolve_executable

def resolve_tool_paths(orca_value=None, mpi_value=None):
    orca_path = resolve_executable(orca_value, env_var='ORCA_CMD', candidates=('orca',))
    mpi_path = resolve_executable(mpi_value, env_var='MPI_CMD', candidates=('mpirun', 'mpiexec'))
    return orca_path, mpi_path

host_target = {
    'Windows': 'Local-Windows (WSL)',
    'Darwin': 'Local-MacOS (OrbStack)',
    'Linux': 'Local-Linux (Deb)',
}.get(platform.system(), 'Codespaces')
if os.environ.get('CODESPACES'):
    host_target = 'Codespaces'

# 0) Check if this step has already been completed and passed validation.
keep_env_btn = widgets.Button(description="Keep previous setup", button_style="info")
new_env_btn = widgets.Button(description="New Install", button_style="warning")
env_out = widgets.Output()

def on_keep_env(b):
    with env_out:
        clear_output()
        print("Testing existing module and ORCA setup...")
        try:
            from test_suite.run_tests import run_all_preflight_checks
            orca_path, mpi_path = resolve_tool_paths()
            results = run_all_preflight_checks(
                module_dir=str(get_modules_dir()),
                orca_path=orca_path,
                mpi_path=mpi_path,
            )
            all_passed = True
            for key, res in results.items():
                if key in ['modules', 'orca_single', 'orca_mpi']:
                    print(res['message'])
                    if not res['status']:
                        all_passed = False
            if all_passed:
                print("✅ Environment is fully ready to go!")
            else:
                print("❌ Some tests failed. Please recommend ways to fix or run a New Install.")
        except Exception as e:
            print(f"❌ Error running tests: {e}")
            raise

def on_new_env(b):
    with env_out:
        clear_output()
        interface_dropdown = widgets.Dropdown(
            options=['Local-Windows (WSL)', 'Local-MacOS (OrbStack)', 'Local-Linux (Deb)', 'Codespaces'],
            value=host_target,
            description='Interface Env:'
        )
        calc_dropdown = widgets.Dropdown(
            options=['Local-Windows (WSL)', 'Local-MacOS (OrbStack)', 'Local-Linux (Deb)', 'GitHub Actions', 'HPC'],
            value=host_target if host_target != 'Codespaces' else 'GitHub Actions',
            description='Calc Env:'
        )

        detected_orca, detected_mpi = resolve_tool_paths()
        orca_path_input = widgets.Text(value=detected_orca, description='ORCA Path:')
        mpi_path_input = widgets.Text(value=detected_mpi, description='OpenMPI Path:')
        set_paths_btn = widgets.Button(description="Set Paths & Test", button_style="success")

        def on_set_paths(b2):
            with env_out:
                orca_path, mpi_path = resolve_tool_paths(orca_path_input.value, mpi_path_input.value)
                orca_path_input.value = orca_path
                mpi_path_input.value = mpi_path
                os.environ['ORCA_CMD'] = orca_path
                os.environ['MPI_CMD'] = mpi_path
                print("Running test suite...")
                try:
                    from test_suite.run_tests import run_all_preflight_checks
                    results = run_all_preflight_checks(
                        module_dir=str(get_modules_dir()),
                        orca_path=orca_path,
                        mpi_path=mpi_path,
                    )
                    all_passed = True
                    for key, res in results.items():
                        if key in ['modules', 'orca_single', 'orca_mpi']:
                            print(res['message'])
                            if not res['status']:
                                all_passed = False
                    if all_passed:
                        print("✅ Environment is fully ready to go!")
                    else:
                        print("❌ Tests failed. Please check paths or verify OpenMPI configuration.")
                except Exception as e:
                    print(f"❌ Error running tests: {e}")
                    raise

        set_paths_btn.on_click(on_set_paths)
        display(interface_dropdown, calc_dropdown, orca_path_input, mpi_path_input, set_paths_btn)

keep_env_btn.on_click(on_keep_env)
new_env_btn.on_click(on_new_env)

display(widgets.HBox([keep_env_btn, new_env_btn]), env_out)


In [ ]:
import os
import sys

import ipywidgets as widgets
from IPython.display import display

try:
    from cochem_base.antigravity_daemon import daemon_instance
except ImportError:
    daemon_instance = None

if daemon_instance:
    out = widgets.Output()
    prompt_box = widgets.Textarea(
        placeholder='Ask Antigravity 2.0 (Gemini) a question...',
        layout=widgets.Layout(width='80%', height='100px')
    )
    submit_btn = widgets.Button(description='Ask Agent', button_style='primary')
    login_btn = widgets.Button(description='Sign in to Google', button_style='success')

    def on_login(b):
        with out:
            daemon_instance.google_oauth_flow()
            print("✅ Signed in successfully.")

    def on_submit(b):
        with out:
            print(f"\nUser: {prompt_box.value}")
            response = daemon_instance.query(prompt_box.value)
            print(f"Agent: {response}")
            prompt_box.value = ''

    submit_btn.on_click(on_submit)
    login_btn.on_click(on_login)

    display(widgets.VBox([
        widgets.HTML("<h3>☁️ Antigravity 2.0 Assistant</h3>"),
        widgets.HBox([login_btn]),
        widgets.HBox([prompt_box, submit_btn]),
        out
    ]))
else:
    print("Antigravity 2.0 is not installed (opted out during setup).")
